# EDA и очистка данных

Смотрим на датасет, который собрал первый DAG (flats_buildings), и проверяем его на три типа проблем: дубликаты, пропуски и выбросы. Функции очистки лежат отдельно, в plugins/steps/cleaning.py, их же использует DAG очистки, поэтому чистка в ноутбуке и в DAG одинаковая.

*Анализ проведён на данных на момент выполнения проекта, цифры в выводах относятся к этому срезу. Правила очистки (медианы, границы межквартильного размаха) пересчитываются при каждом запуске DAG, но если исходные таблицы сильно изменятся, ноутбук нужно будет перезапустить и проверить, подходят ли правила.*

In [ ]:
import os
import sys

import pandas as pd
from sqlalchemy import create_engine

sys.path.append(os.path.abspath("../plugins"))
from steps.cleaning import clean_dataset, drop_duplicate_flats, fix_zero_as_missing, remove_price_outliers, remove_iqr_outliers

PG_USER = os.environ.get("POSTGRES_USER", "mle_user")
PG_PASSWORD = os.environ.get("POSTGRES_PASSWORD", "mle_password")
PG_HOST = os.environ.get("POSTGRES_HOST", "project-db")
PG_PORT = os.environ.get("POSTGRES_PORT", "5432")
PG_DB = os.environ.get("POSTGRES_DB", "buildings_flats")

engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}")
df = pd.read_sql("SELECT * FROM public.flats_buildings;", engine)
df.shape

(141362, 18)

## Дубликаты

Проверяем, есть ли повторяющиеся id квартир.

In [ ]:
print("duplicate flat ids:", df["id"].duplicated().sum())

duplicate flat ids: 0


Дубликатов нет. Функцию удаления дубликатов всё равно оставляем в DAG на случай, если исходные таблицы обновятся.

## Пропуски

Явных пропусков в данных нет, но у части квартир площадь кухни или жилая площадь равна нулю. Квартир без кухни и жилой площади не бывает, поэтому ноль скорее всего означает, что значение просто не заполнили.

In [ ]:
print("kitchen_area == 0:", (df["kitchen_area"] <= 0).sum())
print("living_area == 0:", (df["living_area"] <= 0).sum())
df[["kitchen_area", "living_area", "total_area", "rooms"]].describe()

kitchen_area == 0: 11701
living_area == 0: 18588


,kitchen_area,living_area,total_area,rooms
count,141362.000000,141362.000000,141362.000000,141362.000000
mean,9.001579,31.056948,62.374644,2.129476
std,5.264076,23.968640,40.295864,0.994340
min,0.000000,0.000000,11.000000,1.000000
25%,6.100000,19.000000,39.299999,1.000000
50%,8.800000,29.400000,53.000000,2.000000
75%,10.200000,41.400002,72.000000,3.000000
max,203.000000,700.000000,960.299988,20.000000


Нулевая площадь кухни у 11701 квартиры, нулевая жилая у 18588. Заменяем нули на медиану по квартирам с тем же числом комнат: площадь сильно зависит от числа комнат, и общая медиана исказила бы значения для маленьких и больших квартир.

## Выбросы

Смотрим на цену, высоту потолков и общую площадь.

In [ ]:
df[["price", "ceiling_height", "total_area"]].describe()

,price,ceiling_height,total_area
count,1.413620e+05,141362.000000,141362.000000
mean,1.944162e+07,2.753650,62.374644
std,6.626954e+07,0.223275,40.295864
min,1.100000e+01,2.000000,11.000000
25%,8.900000e+06,2.640000,39.299999
50%,1.185000e+07,2.640000,53.000000
75%,1.695000e+07,2.800000,72.000000
max,9.873738e+09,27.000000,960.299988


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(np.log1p(df["price"]), bins=50)
axes[0].set_title("log1p(price)")
axes[1].hist(df["ceiling_height"], bins=50)
axes[1].set_title("ceiling_height")
plt.tight_layout()
plt.show()

<Figure size 1000x400 with 2 Axes>

Цены лежат в диапазоне от 11 рублей до 9,9 млрд рублей, распределение сильно скошено вправо. Поэтому выбросы цены ищем по межквартильному размаху на логарифме цены, иначе отрезали бы много дорогих, но существующих квартир. С потолками и площадью тоже не всё чисто: есть потолки по 27 м и квартиры на 960 кв.м. Их чистим по обычному межквартильному размаху.

## Очистка

Применяем все функции очистки по очереди и смотрим, сколько строк остаётся после каждого шага.

In [ ]:
cleaned, report = clean_dataset(df)
report

{'input_rows': 141362,
 'after_dedup': 141362,
 'after_missing_fix': 141362,
 'after_price_outliers': 132889,
 'after_iqr_outliers': 123525}

Дубликатов не было, заполнение нулей строки не удаляет. Выбросы цены забрали 8473 строки, выбросы потолков и площади ещё 9364. В итоге осталось 123525 строк из 141362, удалено около 12,6 %.

In [ ]:
print("before:")
print(df[["price", "ceiling_height", "total_area", "kitchen_area", "living_area"]].describe())
print("\nafter:")
print(cleaned[["price", "ceiling_height", "total_area", "kitchen_area", "living_area"]].describe())

before:
              price  ceiling_height     total_area   kitchen_area  \
count  1.413620e+05   141362.000000  141362.000000  141362.000000   
mean   1.944162e+07        2.753650      62.374644       9.001579   
std    6.626954e+07        0.223275      40.295864       5.264076   
min    1.100000e+01        2.000000      11.000000       0.000000   
25%    8.900000e+06        2.640000      39.299999       6.100000   
50%    1.185000e+07        2.640000      53.000000       8.800000   
75%    1.695000e+07        2.800000      72.000000      10.200000   
max    9.873738e+09       27.000000     960.299988     203.000000   

         living_area  
count  141362.000000  
mean       31.056948  
std        23.968640  
min         0.000000  
25%        19.000000  
50%        29.400000  
75%        41.400002  
max       700.000000  

after:
              price  ceiling_height     total_area   kitchen_area  \
count  1.235250e+05   123525.000000  123525.000000  123525.000000   
mean   1.247813e+

После очистки цены лежат в диапазоне от 3,4 до 44,5 млн руб., высота потолков от 2,45 до 3 м, общая площадь от 12,8 до 104 кв.м. Нулевых площадей больше нет. Данные выглядят правдоподобно, можно передавать в DAG и дальше в обучение.